In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

df = pd.read_parquet('demo_1000.parquet')
pf = pq.ParquetFile('demo_1000.parquet')
schema = pf.schema_arrow

# ========== 1. 列名与类型总览 ==========
print(f'总行数: {pf.metadata.num_rows}')
print(f'总列数: {len(schema.names)}')
print()
for i, name in enumerate(schema.names):
    print(f'  [{i:3d}] {name}: {schema.types[i]}')

# ========== 2. Meta 列 ==========
print('\n=== ID & Label ===')
for c in ['user_id', 'item_id', 'label_type', 'label_time', 'timestamp']:
    print(f'  {c}: dtype={df[c].dtype}, nunique={df[c].nunique()}, sample={df[c].iloc[:3].tolist()}')
print(f'  label_type distribution: {df["label_type"].value_counts().to_dict()}')

# ========== 3. User Int 单值特征 ==========
print('\n=== User Int (single-value) ===')
int_single = [c for c in df.columns if c.startswith('user_int') and not isinstance(df[c].iloc[0], (list, np.ndarray))]
for c in sorted(int_single, key=lambda x: int(x.split('_')[-1])):
    s = df[c]
    print(f'  {c}: nunique={s.nunique()}, range=[{s.min()}, {s.max()}]')

# ========== 4. User Int 多值特征 ==========
print('\n=== User Int (multi-value / list) ===')
int_multi = [c for c in df.columns if c.startswith('user_int') and isinstance(df[c].iloc[0], (list, np.ndarray))]
for c in sorted(int_multi, key=lambda x: int(x.split('_')[-1])):
    lens = df[c].apply(lambda x: len(x) if x is not None and hasattr(x, '__len__') else 0)
    vals = df[c].dropna().explode()
    print(f'  {c}: list_len=[{lens.min()},{lens.max()}], vocab_range=[{vals.min()},{vals.max()}], nunique_vals={vals.nunique()}')

# ========== 5. User Dense 特征 ==========
print('\n=== User Dense ===')
dense_cols = sorted([c for c in df.columns if c.startswith('user_dense')], key=lambda x: int(x.split('_')[-1]))
for c in dense_cols:
    s = df[c].dropna()
    first = None
    for v in s:
        if hasattr(v, '__len__') and len(v) > 0:
            first = v
            break
    if first is not None:
        print(f'  {c}: dim={len(first)}, val_range=[{np.min(first):.4f}, {np.max(first):.4f}]')

# ========== 6. Item Int 特征 ==========
print('\n=== Item Int ===')
item_cols = sorted([c for c in df.columns if c.startswith('item_int')], key=lambda x: int(x.split('_')[-1]))
for c in item_cols:
    s = df[c]
    is_list = isinstance(s.iloc[0], (list, np.ndarray))
    if is_list:
        vals = s.dropna().explode()
        print(f'  {c} (list): vocab_range=[{vals.min()},{vals.max()}], nunique={vals.nunique()}')
    else:
        print(f'  {c} (scalar): nunique={s.nunique()}, range=[{s.min()}, {s.max()}]')

# ========== 7. 序列域特征 ==========
for domain in ['domain_a', 'domain_b', 'domain_c', 'domain_d']:
    cols = sorted([c for c in df.columns if c.startswith(domain)], key=lambda x: int(x.split('_')[-1]))
    print(f'\n=== {domain} seq ({len(cols)} cols) ===')
    for c in cols:
        s = df[c].dropna()
        first = None
        for v in s:
            if hasattr(v, '__len__') and len(v) > 0:
                first = v
                break
        if first is None:
            print(f'  {c}: all empty')
            continue
        lens = s.apply(lambda x: len(x) if hasattr(x, '__len__') else 0)
        vals = s.explode()
        is_timestamp = np.max(first) > 1e9
        tag = ' **TIMESTAMP**' if is_timestamp else ''
        print(f'  {c}: seq_len=[{lens.min()},{lens.max()}], vocab_range=[{vals.min()},{vals.max()}], nunique_vals={vals.nunique()}{tag}')

# ========== 8. Int-Dense 对齐关系检查 ==========
print('\n=== Int-Dense Aligned Pairs ===')
for fid in [62, 63, 64, 65, 66, 89, 90, 91]:
    int_col = f'user_int_feats_{fid}'
    dense_col = f'user_dense_feats_{fid}'
    if int_col in df.columns and dense_col in df.columns:
        int_lens = df[int_col].apply(lambda x: len(x) if hasattr(x, '__len__') else 0)
        dense_lens = df[dense_col].apply(lambda x: len(x) if hasattr(x, '__len__') else 0)
        aligned = (int_lens == dense_lens).mean()
        print(f'  fid {fid}: int_len mean={int_lens.mean():.1f}, dense_len mean={dense_lens.mean():.1f}, aligned_ratio={aligned:.2%}')


In [ ]:
from datetime import datetime

for i in range(5):
    lt = df['label_time'].iloc[i]
    ts = df['timestamp'].iloc[i]
    lb = df['label_type'].iloc[i]
    lt_dt = datetime.fromtimestamp(lt)
    ts_dt = datetime.fromtimestamp(ts)
    delay = lt - ts
    print(f'row {i}: label_type={lb}, timestamp={ts} ({ts_dt}), label_time={lt} ({lt_dt}), delay={delay}s ({delay/3600:.1f}h)')

In [ ]:
import sys
sys.path.insert(0, '/home/apulis-dev/code/qq-algo/sample_code')

import json, os
import torch
from dataset import FeatureSchema, get_pcvr_data, NUM_TIME_BUCKETS, PCVRParquetDataset
from model import PCVRHyFormer, ModelInput
from utils import set_seed

In [ ]:
set_seed(42)

data_dir = '/home/apulis-dev/code/qq-algo'
schema_path = os.path.join(data_dir, 'schema.json')

train_loader, valid_loader, pcvr_dataset = get_pcvr_data(
    data_dir=data_dir,
    schema_path=schema_path,
    batch_size=256,
    valid_ratio=0.1,
    seed=42,
    num_workers=0,
)

In [ ]:
def build_feature_specs(schema, per_position_vocab_sizes):
    specs = []
    for fid, offset, length in schema.entries:
        vs = max(per_position_vocab_sizes[offset:offset + length])
        specs.append((vs, offset, length))
    return specs

user_int_feature_specs = build_feature_specs(
    pcvr_dataset.user_int_schema, pcvr_dataset.user_int_vocab_sizes)
item_int_feature_specs = build_feature_specs(
    pcvr_dataset.item_int_schema, pcvr_dataset.item_int_vocab_sizes)

# run.sh uses --ns_groups_json "" (empty), so each feature is its own group
user_ns_groups = [[i] for i in range(len(pcvr_dataset.user_int_schema.entries))]
item_ns_groups = [[i] for i in range(len(pcvr_dataset.item_int_schema.entries))]
print(f'user_ns_groups: {len(user_ns_groups)} groups (each feature alone)')
print(f'item_ns_groups: {len(item_ns_groups)} groups (each feature alone)')

In [ ]:
pcvr_dataset.seq_domain_vocab_sizes

In [ ]:
model = PCVRHyFormer(
    user_int_feature_specs=user_int_feature_specs,
    item_int_feature_specs=item_int_feature_specs,
    user_dense_dim=pcvr_dataset.user_dense_schema.total_dim,
    item_dense_dim=pcvr_dataset.item_dense_schema.total_dim,
    seq_vocab_sizes=pcvr_dataset.seq_domain_vocab_sizes,
    user_ns_groups=user_ns_groups,
    item_ns_groups=item_ns_groups,
    d_model=64,
    emb_dim=64,
    num_queries=2,
    num_hyformer_blocks=2,
    num_heads=4,
    seq_encoder_type='transformer',
    hidden_mult=4,
    dropout_rate=0.01,
    seq_top_k=50,
    seq_causal=False,
    action_num=1,
    num_time_buckets=NUM_TIME_BUCKETS,
    rank_mixer_mode='full',
    use_rope=False,
    rope_base=10000.0,
    emb_skip_threshold=1000000,
    seq_id_threshold=10000,
    ns_tokenizer_type='rankmixer',
    user_ns_tokens=5,
    item_ns_tokens=2,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'device: {device}')
print(f'num_ns={model.num_ns}, num_sequences={model.num_sequences}')
T = 2 * model.num_sequences + model.num_ns
print(f'T = num_queries*num_sequences + num_ns = {T}')
print(f'd_model % T == 0: {64 % T == 0}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
print(model)

In [ ]:
model.eval()

infer_dataset = PCVRParquetDataset(
    parquet_path=data_dir,
    schema_path=schema_path,
    batch_size=1,
    shuffle=False,
    buffer_batches=0,
    is_training=False,
)
batch = next(iter(infer_dataset))
device_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

seq_data = {}
seq_lens = {}
seq_time_buckets = {}
for domain in pcvr_dataset.seq_domains:
    seq_data[domain] = device_batch[domain]
    seq_lens[domain] = device_batch[f'{domain}_len']
    B = seq_data[domain].shape[0]
    L = seq_data[domain].shape[2]
    seq_time_buckets[domain] = device_batch.get(
        f'{domain}_time_bucket',
        torch.zeros(B, L, dtype=torch.long, device=device))

inputs = ModelInput(
    user_int_feats=device_batch['user_int_feats'],
    item_int_feats=device_batch['item_int_feats'],
    user_dense_feats=device_batch['user_dense_feats'],
    item_dense_feats=device_batch['item_dense_feats'],
    seq_data=seq_data,
    seq_lens=seq_lens,
    seq_time_buckets=seq_time_buckets,
)

with torch.no_grad():
    logits = model(inputs)

print(f'Input batch size: {inputs.user_int_feats.shape[0]}')
print(f'Output logits shape: {logits.shape}')
print(f'Logits sample: {logits[:5].squeeze()}')
print(f'Probabilities: {torch.sigmoid(logits[:5].squeeze())}')

In [ ]:
model.clsfier